In [0]:
%run /Users/shreya.dani22@gmail.com/Final_project/Logging

In [0]:
silver_dim_products_all = spark.table("retail_catalog.retail_schema.silver_dim_product_all")

In [0]:
silver_dim_products = spark.table("retail_catalog.retail_schema.silver_dim_product_all").filter("is_current = true")

In [0]:
silver_fact_inventory = spark.table("retail_catalog.retail_schema.silver_fact_inventory")

In [0]:
silver_fact_product_metrics = spark.table("retail_catalog.retail_schema.silver_fact_product_metrics")

In [0]:
silver_fact_reviews = spark.table("retail_catalog.retail_schema.silver_fact_reviews")

In [0]:
log_message("INFO", "Gold layer notebook started")
log_message("INFO", "All silver tables loaded: silver_dim_product_all, silver_fact_inventory, silver_fact_product_metrics, silver_fact_reviews")

INFO: Gold layer notebook started
INFO: All silver tables loaded: silver_dim_product_all, silver_fact_inventory, silver_fact_product_metrics, silver_fact_reviews


**Category Wise product count**

In [0]:
from pyspark.sql.functions import count

log_message("INFO", "Analyzing category product distribution")

silver_dim_products.groupBy("category") \
    .agg(count("product_id").alias("total_products")) \
    .orderBy("total_products", ascending=False) \
    .limit(10).display()

log_message("INFO", "Category product distribution completed")

INFO: Analyzing category product distribution


category,total_products
kitchen-accessories,30
groceries,27
sports-accessories,17
smartphones,16
mobile-accessories,14
mens-watches,6
beauty,5
womens-watches,5
mens-shoes,5
fragrances,5


INFO: Category product distribution completed


**Average price and average discount**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import round as spark_round

log_message("INFO", "Analyzing pricing strategy by category and price segment")

df_seg = silver_dim_products.withColumn(
    "price_segment",
    when(silver_dim_products.price < 50, "Low")
    .when((silver_dim_products.price >= 50) & (silver_dim_products.price < 500), "Medium")
    .otherwise("High")
)

gold_pricing_strategy = df_seg.groupBy("category", "price_segment") \
    .agg(
        spark_round(avg("price"), 2).alias("avg_price"),
        spark_round(avg("discountPercentage"), 2).alias("avg_discount"),
        count("*").alias("product_count")
    ).orderBy("category")

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/pricing_strategy"


gold_pricing_strategy.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_pricing_strategy")
display(gold_pricing_strategy.limit(10))

log_message("INFO", "gold_pricing_strategy saved to gold layer")

INFO: Analyzing pricing strategy by category and price segment


category,price_segment,avg_price,avg_discount,product_count
beauty,Low,13.39,12.42,5
fragrances,Low,49.99,1.89,1
fragrances,Medium,92.49,11.56,4
furniture,Medium,399.99,10.55,2
furniture,High,1733.32,10.59,3
groceries,Low,5.47,9.22,27
home-decoration,Low,33.74,9.07,4
home-decoration,Medium,59.99,10.41,1
kitchen-accessories,Low,15.2,9.54,29
kitchen-accessories,Medium,89.99,12.13,1


INFO: gold_pricing_strategy saved to gold layer


**Comparision of warranty info, avg price and discount**

In [0]:
from pyspark.sql.functions import round as spark_round, when, col

warranty_order = (
    when(col("warrantyInformation") == "0", 0)
    .when(col("warrantyInformation") == "1 week", 1)
    .when(col("warrantyInformation") == "1 month", 2)
    .when(col("warrantyInformation") == "3 months", 3)
    .when(col("warrantyInformation") == "6 months", 4)
    .when(col("warrantyInformation") == "1 year", 5)
    .when(col("warrantyInformation") == "2 year", 6)
    .when(col("warrantyInformation") == "3 year", 7)
    .when(col("warrantyInformation") == "5 year", 8)
    .when(col("warrantyInformation") == "lifetime", 9)
    .otherwise(10)
)

gold_warranty_analysis = silver_dim_products.groupBy("warrantyInformation") \
    .agg(
        spark_round(avg("price"), 2).alias("avg_price"),
        spark_round(avg("discountPercentage"), 2).alias("avg_discount"),
        count("*").alias("product_count")
    ).withColumn("sort_order", warranty_order) \
    .orderBy("sort_order") \
    .drop("sort_order")

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/warranty_analysis"

gold_warranty_analysis.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_warranty_analysis")
display(gold_warranty_analysis.limit(10))

log_message("INFO", "gold_warranty_analysis saved to gold layer")

warrantyInformation,avg_price,avg_discount,product_count
0,233.18,11.05,12
1 week,148.53,10.62,13
1 month,4049.52,9.7,24
3 months,639.15,11.98,19
6 months,942.96,11.8,16
1 year,1204.7,9.83,31
2 year,2631.81,10.69,20
3 year,1312.88,9.89,28
5 year,2596.92,11.19,14
lifetime,726.58,10.37,17


INFO: gold_warranty_analysis saved to gold layer


**Price change and discount change**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, round as spark_round

windowSpec = Window.partitionBy("product_id").orderBy("effective_start_date")

gold_price_trend = silver_dim_products_all.withColumn("prev_price", lag("price").over(windowSpec)) \
    .withColumn("price_change", spark_round(col("price") - col("prev_price"), 2)) \
    .withColumn("prev_discount", lag("discountPercentage").over(windowSpec)) \
    .withColumn("discount_change", spark_round(col("discountPercentage") - col("prev_discount"), 2))

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/price_trend"

gold_price_trend.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_price_trend")
display(gold_price_trend.limit(10))

log_message("INFO", "gold_price_trend saved to gold layer")

product_id,title,category,brand,sku,price,discountPercentage,warrantyInformation,shippingInformation,returnPolicy,minimumOrderQuantity,ingestion_time,effective_start_date,effective_end_date,is_current,prev_price,price_change,prev_discount,discount_change
1,Essence Mascara Lash Princess,beauty,essence,BEA-ESS-ESS-001,9.99,10.48,1 week,5,0,48,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
2,Eyeshadow Palette with Mirror,beauty,glamour beauty,BEA-GLA-EYE-002,19.99,18.19,1 year,14,7,20,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
3,Powder Canister,beauty,velvet touch,BEA-VEL-POW-003,14.99,9.84,3 months,2,0,22,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
4,Red Lipstick,beauty,chic cosmetics,BEA-CHI-LIP-004,12.99,12.16,3 year,7,7,40,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
5,Red Nail Polish,beauty,nail couture,BEA-NAI-NAI-005,8.99,11.44,1 month,1,0,22,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
6,Calvin Klein CK One,fragrances,calvin klein,FRA-CAL-CAL-006,49.99,1.89,1 week,1,90,9,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
7,Chanel Coco Noir Eau De,fragrances,chanel,FRA-CHA-CHA-007,129.99,16.51,3 year,1,0,1,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
8,Dior J'adore,fragrances,dior,FRA-DIO-DIO-008,89.99,14.72,1 week,14,7,10,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
9,Dolce Shine Eau de,fragrances,dolce & gabbana,FRA-DOL-DOL-009,69.99,0.62,3 year,30,7,2,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null
10,Gucci Bloom Eau de,fragrances,gucci,FRA-GUC-GUC-010,79.99,14.39,6 months,1,0,2,2026-04-06T19:57:28.815746Z,2026-04-06T19:57:28.815746Z,null,true,null,null,null,null


INFO: gold_price_trend saved to gold layer


**Inventory Summury**

In [0]:
from pyspark.sql.functions import count, avg, round as spark_round

log_message("INFO", "Analyzing inventory alerts by category and availability")

df_joined = silver_fact_inventory.join(silver_dim_products, "product_id")

gold_inventory_summary = df_joined.groupBy("category", "availabilityStatus") \
    .agg(
        count("*").alias("product_count"),
        spark_round(avg("stock"), 2).alias("avg_stock"),
        spark_round(avg("price"), 2).alias("avg_price")
    )

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/inventory_summary"

gold_inventory_summary.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_inventory_summary")
display(gold_inventory_summary.limit(10))

log_message("INFO", "gold_inventory_summary saved to gold layer")

INFO: Analyzing inventory alerts by category and availability


category,availabilityStatus,product_count,avg_stock,avg_price
laptops,In Stock,5,51.6,1559.99
sunglasses,In Stock,4,49.25,26.99
womens-shoes,In Stock,3,76.67,53.32
mens-shirts,In Stock,4,76.25,29.49
womens-watches,Low Stock,1,4.0,10999.99
home-decoration,In Stock,5,44.0,38.99
womens-bags,In Stock,5,60.0,175.99
womens-jewellery,In Stock,2,63.5,27.49
motorcycle,In Stock,4,45.0,7749.99
womens-dresses,Low Stock,1,6.0,49.99


INFO: gold_inventory_summary saved to gold layer


**Low Stock**

In [0]:
log_message("INFO", "Identifying low stock products (stock < 10)")

gold_low_stock_alert = df_joined.filter(df_joined.stock < 10).select(
    "product_id", "title", "category", "brand",
    "stock", "price", "availabilityStatus"
)

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/low_stock_alert"

gold_low_stock_alert.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_low_stock_alert")
display(gold_low_stock_alert.limit(10))

log_message("INFO", "gold_low_stock_alert saved to gold layer")

INFO: Identifying low stock products (stock < 10)


product_id,title,category,brand,stock,price,availabilityStatus
184,Tropical Earring,womens-jewellery,unknown,1,19.99,Low Stock
88,Nike Air Jordan 1 Red And Black,mens-shoes,nike,7,149.99,In Stock
185,Black & Brown Slipper,womens-shoes,comfort trends,3,19.99,Low Stock
180,Dress Pea,womens-dresses,unknown,6,49.99,Low Stock
117,Sportbike Motorcycle,motorcycle,speedmaster,0,7499.99,Out of Stock
86,Man Short Sleeve Shirt,mens-shirts,casual comfort,2,19.99,Low Stock
189,Red Shoes,womens-shoes,fashion express,7,34.99,Low Stock
155,Classic Sun Glasses,sunglasses,fashion shades,1,24.99,Low Stock
136,Vivo X21,smartphones,vivo,7,499.99,In Stock
153,Volleyball,sports-accessories,unknown,0,11.99,Out of Stock


INFO: gold_low_stock_alert saved to gold layer


**Overstock**

In [0]:
log_message("INFO", "Identifying overstock products (stock > 2x minimumOrderQuantity)")

gold_overstock_alert = df_joined.filter(
    df_joined.stock > df_joined.minimumOrderQuantity * 2
).select(
    "product_id", "title", "category",
    "stock", "minimumOrderQuantity", "price"
)

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/overstock_alert"

gold_overstock_alert.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_overstock_alert")
display(gold_overstock_alert.limit(10))

log_message("INFO", "gold_overstock_alert saved to gold layer")

INFO: Identifying overstock products (stock > 2x minimumOrderQuantity)


product_id,title,category,stock,minimumOrderQuantity,price
167,300 Touring,vehicle,54,1,28999.99
112,TV Studio Camera Pedestal,mobile-accessories,15,1,499.99
113,Generic Motorcycle,motorcycle,34,1,3999.99
188,Pampi Shoes,womens-shoes,49,12,29.99
190,IWC Ingenieur Automatic Steel,womens-watches,90,1,4999.99
84,Gigabyte Aorus Men Tshirt,mens-shirts,90,16,24.99
187,Golden Shoes Woman,womens-shoes,88,7,49.99
116,Scooter Motorcycle,motorcycle,84,1,2999.99
181,Marni Red & Black Suit,womens-dresses,62,2,179.99
161,Samsung Galaxy Tab White,tablets,92,5,349.99


INFO: gold_overstock_alert saved to gold layer


**Stock change**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

windowSpec = Window.partitionBy("product_id").orderBy("updated_date")

gold_inventory_trend = silver_fact_inventory.withColumn(
    "prev_stock", lag("stock").over(windowSpec)
).withColumn(
    "stock_change", col("stock") - col("prev_stock")
)

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/inventory_trend"

gold_inventory_trend.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_inventory_trend")
display(gold_inventory_trend.limit(10))

log_message("INFO", "gold_inventory_trend saved to gold layer")

product_id,stock,availabilityStatus,updated_at,updated_date,prev_stock,stock_change
1,99,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
2,34,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
3,89,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
4,91,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
5,79,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
6,29,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
7,58,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
8,98,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
9,4,Low Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null
10,91,In Stock,2025-04-30T09:41:02.053Z,2025-04-30,null,null


INFO: gold_inventory_trend saved to gold layer


**Demand**

In [0]:
log_message("INFO", "Generating smart demand alerts")

df_all = silver_fact_inventory.alias("inv") \
    .join(silver_dim_products.alias("prod"), "product_id") \
    .join(silver_fact_product_metrics.alias("met"), "product_id")

gold_smart_demand_alerts = df_all.select(
    col("product_id"),
    col("prod.title"),
    col("prod.category"),
    col("prod.brand"),
    col("inv.stock"),
    col("inv.availabilityStatus"),
    col("prod.price"),
    col("met.rating")
).withColumn(
    "alert_type",
    when((col("stock") < 10) & (col("rating") > 4), "High Demand - Restock Urgently")
    .when((col("stock") > 100) & (col("rating") < 3), "Overstock - Low Demand")
    .otherwise("Normal")
)

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/smart_demand_alert"

gold_smart_demand_alerts.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_smart_demand_alerts")
display(gold_smart_demand_alerts.limit(10))

log_message("INFO", "gold_smart_demand_alerts saved to gold layer")

INFO: Generating smart demand alerts


product_id,title,category,brand,stock,availabilityStatus,price,rating,alert_type
167,300 Touring,vehicle,chrysler,54,In Stock,28999.99,4.05,Normal
112,TV Studio Camera Pedestal,mobile-accessories,provision,15,In Stock,499.99,2.78,Normal
113,Generic Motorcycle,motorcycle,generic motors,34,In Stock,3999.99,4.91,Normal
184,Tropical Earring,womens-jewellery,unknown,1,Low Stock,19.99,4.4,High Demand - Restock Urgently
188,Pampi Shoes,womens-shoes,pampi,49,In Stock,29.99,3.05,Normal
190,IWC Ingenieur Automatic Steel,womens-watches,iwc,90,In Stock,4999.99,2.93,Normal
84,Gigabyte Aorus Men Tshirt,mens-shirts,gigabyte,90,In Stock,24.99,3.18,Normal
119,Olay Ultra Moisture Shea Butter Body Wash,skin-care,olay,34,In Stock,12.99,4.51,Normal
187,Golden Shoes Woman,womens-shoes,fashion diva,88,In Stock,49.99,3.26,Normal
116,Scooter Motorcycle,motorcycle,scootmaster,84,In Stock,2999.99,2.53,Normal


INFO: gold_smart_demand_alerts saved to gold layer


**Average rating per product**

In [0]:
from pyspark.sql.functions import round as spark_round

log_message("INFO", "Analyzing reviews by category")

joined_reviews = silver_fact_reviews.join(silver_dim_products, "product_id")

gold_reviews_summary = joined_reviews.groupBy("product_id", "title", "category", "brand").agg(
    count("*").alias("review_count"),
    spark_round(avg("review_rating"), 2).alias("avg_rating")
).orderBy("product_id")

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/reviews_summary"
gold_reviews_summary.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_reviews_summary")
display(gold_reviews_summary.limit(10))

log_message("INFO", "gold_reviews_summary saved to gold layer")

INFO: Analyzing reviews by category


product_id,title,category,brand,review_count,avg_rating
1,Essence Mascara Lash Princess,beauty,essence,6,4.0
2,Eyeshadow Palette with Mirror,beauty,glamour beauty,6,3.33
3,Powder Canister,beauty,velvet touch,6,3.33
4,Red Lipstick,beauty,chic cosmetics,6,4.67
5,Red Nail Polish,beauty,nail couture,6,2.67
6,Calvin Klein CK One,fragrances,calvin klein,6,3.0
7,Chanel Coco Noir Eau De,fragrances,chanel,6,4.67
8,Dior J'adore,fragrances,dior,6,4.33
9,Dolce Shine Eau de,fragrances,dolce & gabbana,6,4.33
10,Gucci Bloom Eau de,fragrances,gucci,6,3.33


INFO: gold_reviews_summary saved to gold layer


   
**Rating vs Price Correlation**

In [0]:
from pyspark.sql.functions import corr, round as spark_round

log_message("INFO", "Starting Rating vs Price Correlation analysis")

df_price_rating = silver_dim_products.join(silver_fact_product_metrics, "product_id")

gold_rating_vs_price = df_price_rating.groupBy("category").agg(
    spark_round(avg("price"), 2).alias("avg_price"),
    spark_round(avg("rating"), 2).alias("avg_rating"),
    spark_round(corr("price", "rating"), 4).alias("price_rating_correlation"),
    count("*").alias("product_count")
).orderBy("category")

target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/rating_vs_price"

gold_rating_vs_price.write.mode("overwrite").option("path", target_path)\
    .saveAsTable("retail_catalog.retail_schema.gold_rating_vs_price")
display(gold_rating_vs_price.limit(10))

log_message("INFO", "gold_rating_vs_price saved to gold layer")

INFO: Starting Rating vs Price Correlation analysis


category,avg_price,avg_rating,price_rating_correlation,product_count
beauty,13.39,3.75,-0.1868,5
fragrances,83.99,3.83,0.0491,5
furniture,1199.99,4.01,0.3122,5
groceries,5.47,3.91,0.0597,27
home-decoration,38.99,3.78,-0.0543,5
kitchen-accessories,17.69,3.82,0.2444,30
laptops,1559.99,3.62,0.2163,5
mens-shirts,27.59,3.18,0.6008,5
mens-shoes,109.99,4.6,0.5358,5
mens-watches,8098.32,3.66,-0.5728,6


INFO: gold_rating_vs_price saved to gold layer


   
**Time-based Review Trend**

In [0]:
from pyspark.sql.functions import date_format

log_message("INFO", "Starting Time-based Review Trend analysis")

gold_review_trend = joined_reviews.groupBy(
    date_format("review_date", "yyyy-MM").alias("review_month")
).agg(
    count("*").alias("review_count"),
    spark_round(avg("review_rating"), 2).alias("avg_rating")
).orderBy("review_month")


target_path = "abfss://gold@finalprojstore.dfs.core.windows.net/review_trend"

gold_review_trend.write.mode("overwrite").option("path", target_path).saveAsTable("retail_catalog.retail_schema.gold_review_trend")
   
display(gold_review_trend.limit(10))

log_message("INFO", "gold_review_trend saved to gold layer")

INFO: Starting Time-based Review Trend analysis


review_month,review_count,avg_rating
2025-04,1164,3.7


INFO: gold_review_trend saved to gold layer


   
**Logging Summary**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

log_message("INFO", "Gold layer analysis completed")

# Save log records to a DataFrame for review
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("level", StringType(), True),
    StructField("message", StringType(), True)
])
df_logs = spark.createDataFrame(log_records, schema)

df_logs.write.format("delta") \
    .mode("append") \
    .save("abfss://gold@finalprojstore.dfs.core.windows.net/logs/ext_loc_catalog/")

log_message("INFO", "Logs saved to abfss://gold@finalprojstore.dfs.core.windows.net/logs/ext_loc_catalog/")
print("\n--- Log Summary ---")
display(df_logs.limit(10))

INFO: Gold layer analysis completed
INFO: Logs saved to abfss://gold@finalprojstore.dfs.core.windows.net/logs/ext_loc_catalog/

--- Log Summary ---


timestamp,level,message
2026-04-07T09:58:55.957813Z,INFO,Gold layer notebook started
2026-04-07T09:58:55.957938Z,INFO,"All silver tables loaded: silver_dim_product_all, silver_fact_inventory, silver_fact_product_metrics, silver_fact_reviews"
2026-04-07T09:58:56.16216Z,INFO,Analyzing category product distribution
2026-04-07T09:58:56.891317Z,INFO,Category product distribution completed
2026-04-07T09:58:57.074076Z,INFO,Analyzing pricing strategy by category and price segment
2026-04-07T09:59:00.784194Z,INFO,gold_pricing_strategy saved to gold layer
2026-04-07T09:59:03.960173Z,INFO,gold_warranty_analysis saved to gold layer
2026-04-07T09:59:07.037867Z,INFO,gold_price_trend saved to gold layer
2026-04-07T09:59:07.409292Z,INFO,Analyzing inventory alerts by category and availability
2026-04-07T09:59:10.259692Z,INFO,gold_inventory_summary saved to gold layer
